# HDB Singapore resale — preliminary EDA

Exploratory analysis of **`data/train.csv`**: data quality, target `resale_price`, time trends, flat and location attributes, amenities, correlations, and geographic patterns.

**Working directory:** If the kernel cwd is the project root, data loads from `data/train.csv`. If cwd is `notebook/`, loads from `../data/train.csv`.



In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 12

RNG = np.random.RandomState(42)

_cwd = Path.cwd()
ROOT = _cwd.parent if _cwd.name == "notebook" else _cwd
DATA_PATH = ROOT / "data" / "train.csv"
print(f"Loading: {DATA_PATH.resolve()}")

df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")



In [ ]:
mem_mb = df.memory_usage(deep=True).sum() / 1e6
print(f"Approx. deep memory usage: {mem_mb:.1f} MB")
display(df.head())
df.info(verbose=False)
display(df.describe(include="all").T.head(45))



## 1. Data quality

Missing values, duplicate keys, and parsing transaction dates.



In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
miss_df = pd.DataFrame({"missing": missing, "pct": missing_pct})
miss_nonzero = miss_df[miss_df["missing"] > 0].sort_values("missing", ascending=False)
print(f"Columns with any missing: {len(miss_nonzero)}")
display(miss_nonzero.head(25))

if len(miss_nonzero):
    top = miss_nonzero.head(20)
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(x=top["pct"].values, y=top.index.astype(str), ax=ax)
    ax.set_xlabel("Missing %")
    ax.set_title("Top columns by missing percentage")
    plt.tight_layout()
    plt.show()

dup_id = df["id"].duplicated().sum()
print(f"Duplicate id rows: {dup_id}")



In [ ]:
df["Tranc_period"] = pd.to_datetime(df["Tranc_YearMonth"], format="%Y-%m")

for c in df.select_dtypes(include="object").columns:
    if df[c].eq("").any():
        df[c] = df[c].replace("", np.nan)



## 2. Target variable: `resale_price`

Distribution, skewness, log view, boxplot, and IQR-based outlier counts (report only).



In [ ]:
target = df["resale_price"]
print(target.describe())
print(f"Skewness: {target.skew():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(target, kde=True, ax=axes[0])
axes[0].set_title("resale_price")
sns.boxplot(x=target, ax=axes[1])
axes[1].set_title("resale_price boxplot")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(np.log1p(target), kde=True, ax=ax)
ax.set_title("log1p(resale_price)")
plt.tight_layout()
plt.show()

q1, q3 = target.quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
below = (target < low).sum()
above = (target > high).sum()
print(f"IQR fences: [{low:,.0f}, {high:,.0f}] — below: {below:,}, above: {above:,}")



## 3. Time dimension

Median/mean price and transaction volume over `Tranc_YearMonth`; median price by calendar month.



In [ ]:
monthly = df.groupby("Tranc_period", as_index=False).agg(
    median_price=("resale_price", "median"),
    mean_price=("resale_price", "mean"),
    n=("resale_price", "count"),
)

fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
axes[0].plot(monthly["Tranc_period"], monthly["median_price"], label="median")
axes[0].plot(monthly["Tranc_period"], monthly["mean_price"], alpha=0.7, label="mean")
axes[0].set_ylabel("SGD")
axes[0].legend()
axes[0].set_title("Resale price over time")
axes[1].bar(monthly["Tranc_period"], monthly["n"], width=20)
axes[1].set_ylabel("Transaction count")
axes[1].set_title("Volume over time")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

month_median = df.groupby("Tranc_Month")["resale_price"].median()
fig, ax = plt.subplots(figsize=(10, 4))
month_median.reindex(range(1, 13)).plot(kind="bar", ax=ax, color="steelblue")
ax.set_xlabel("Month")
ax.set_ylabel("Median resale_price")
ax.set_title("Median price by calendar month (all years pooled)")
plt.tight_layout()
plt.show()



## 4. Categorical features

Median price by `flat_type`, `town`, `planning_area`, and top categories for `flat_model` / `storey_range`.



In [ ]:
FLAT_ORDER = [
    "1 ROOM",
    "2 ROOM",
    "3 ROOM",
    "4 ROOM",
    "5 ROOM",
    "EXECUTIVE",
    "MULTI-GENERATION",
    "STUDIO APARTMENT",
]


def median_bar_by_order(series_name, order_list=None, title=None):
    sub = df[[series_name, "resale_price"]].dropna(subset=[series_name])
    med = sub.groupby(series_name)["resale_price"].median()
    cnt = sub[series_name].value_counts()
    if order_list:
        cats = [c for c in order_list if c in med.index]
        med = med.reindex(cats)
    else:
        med = med.sort_values(ascending=False).head(20)
    plot_df = pd.DataFrame({"median": med, "count": cnt.reindex(med.index)})
    plot_df = plot_df.dropna(subset=["median"])
    bar_df = plot_df.rename_axis(series_name).reset_index()
    bar_df[series_name] = bar_df[series_name].astype(str)
    fig, ax = plt.subplots(figsize=(10, max(4, len(bar_df) * 0.35)))
    sns.barplot(data=bar_df, x="median", y=series_name, ax=ax, orient="h")
    ax.set_xlabel("Median resale_price (SGD)")
    ax.set_title(title or series_name)
    plt.tight_layout()
    plt.show()


median_bar_by_order("flat_type", order_list=FLAT_ORDER, title="Median price by flat_type")
median_bar_by_order("town", title="Median price by town (top 20 by median)")
median_bar_by_order("planning_area", title="Median price by planning_area (top 20 by median)")

for col in ["flat_model", "storey_range"]:
    sub = df.groupby(col)["resale_price"].agg(["median", "count"]).reset_index()
    sub = sub.sort_values("count", ascending=False).head(15)
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(data=sub, x="median", y=col, ax=ax, orient="h")
    ax.set_xlabel("Median resale_price (SGD)")
    ax.set_title(f"Median price — top 15 {col} by frequency")
    plt.tight_layout()
    plt.show()



## 5. Numeric features & price per sqm

Distributions and relationships to price; `price_psm = resale_price / floor_area_sqm`.



In [ ]:
df["price_psm"] = df["resale_price"] / df["floor_area_sqm"]

num_cols = [
    "floor_area_sqm",
    "hdb_age",
    "mid_storey",
    "max_floor_lvl",
    "total_dwelling_units",
    "price_psm",
]
num_cols = [c for c in num_cols if c in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.ravel()
for i, c in enumerate(num_cols[:6]):
    sns.histplot(df[c].dropna(), kde=True, ax=axes[i])
    axes[i].set_title(c)
for j in range(len(num_cols), 6):
    axes[j].set_visible(False)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.scatterplot(data=df.sample(min(8000, len(df)), random_state=42), x="floor_area_sqm", y="resale_price", alpha=0.25, ax=axes[0])
axes[0].set_title("floor_area_sqm vs resale_price (8k sample)")
sns.scatterplot(data=df.sample(min(8000, len(df)), random_state=43), x="hdb_age", y="resale_price", alpha=0.25, ax=axes[1])
axes[1].set_title("hdb_age vs resale_price (8k sample)")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df["price_psm"].clip(upper=df["price_psm"].quantile(0.99)), kde=True, ax=ax)
ax.set_title("price_psm (clipped at 99th pct for display)")
plt.tight_layout()
plt.show()



In [ ]:
sub = df.sample(min(15000, len(df)), random_state=44)
plt.figure(figsize=(8, 6))
plt.hexbin(sub["floor_area_sqm"], sub["resale_price"], gridsize=45, cmap="viridis", mincnt=1)
plt.colorbar(label="count")
plt.xlabel("floor_area_sqm")
plt.ylabel("resale_price")
plt.title("Hexbin: floor_area vs price (15k sample)")
plt.tight_layout()
plt.show()



## 6. Amenities & accessibility

Distances and within-radius flags vs median price.



In [ ]:
dist_cols = [
    "mrt_nearest_distance",
    "Mall_Nearest_Distance",
    "Hawker_Nearest_Distance",
    "bus_stop_nearest_distance",
    "pri_sch_nearest_distance",
]
dist_cols = [c for c in dist_cols if c in df.columns]

for c in dist_cols:
    sns.histplot(df[c].dropna(), kde=False, bins=40)
    plt.title(f"Distribution: {c}")
    plt.tight_layout()
    plt.show()


def median_by_distance_deciles(col):
    if col not in df.columns or df[col].notna().sum() < 100:
        return
    valid = df[[col, "resale_price"]].dropna().copy()
    valid["decile"] = pd.qcut(valid[col], q=10, duplicates="drop")
    g = valid.groupby("decile", observed=True)["resale_price"].median()
    fig, ax = plt.subplots(figsize=(10, 4))
    g.plot(kind="bar", ax=ax, color="coral")
    ax.set_title(f"Median resale_price by decile of {col}")
    ax.set_xlabel("Decile (1 = nearest)")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


for c in dist_cols:
    median_by_distance_deciles(c)

flag_cols = [
    "Mall_Within_500m",
    "Mall_Within_1km",
    "Hawker_Within_500m",
    "Mall_Within_2km",
]
flag_cols = [c for c in flag_cols if c in df.columns]
for c in flag_cols:
    if df[c].notna().sum() == 0:
        continue
    tmp = df[[c, "resale_price"]].dropna()
    fig, ax = plt.subplots(figsize=(6, 4))
    sns.boxplot(data=tmp, x=c, y="resale_price", ax=ax)
    ax.set_title(f"resale_price vs {c}")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()



## 7. Correlation (focused numeric subset)

Pearson correlation heatmap for interpretable numeric columns — not the full wide matrix.



In [ ]:
corr_cols = [
    "resale_price",
    "floor_area_sqm",
    "hdb_age",
    "mid_storey",
    "max_floor_lvl",
    "lease_commence_date",
    "total_dwelling_units",
    "price_psm",
    "mrt_nearest_distance",
    "Mall_Nearest_Distance",
    "Hawker_Nearest_Distance",
    "bus_stop_nearest_distance",
    "pri_sch_nearest_distance",
    "sec_sch_nearest_dist",
    "hawker_food_stalls",
    "hawker_market_stalls",
]
corr_cols = [c for c in corr_cols if c in df.columns]
num_df = df[corr_cols].apply(pd.to_numeric, errors="coerce")
cm = num_df.corr(method="pearson")

plt.figure(figsize=(11, 9))
sns.heatmap(cm, annot=False, cmap="RdBu_r", center=0, vmin=-1, vmax=1)
plt.title("Pearson correlation (selected numeric features)")
plt.tight_layout()
plt.show()

rp = cm["resale_price"].drop("resale_price").sort_values(key=abs, ascending=False)
print("Correlation with resale_price (absolute rank):")
display(rp.head(20))



## 8. Geography

Sample scatter / hexbin of coordinates coloured by price.



In [ ]:
geo = df[["Latitude", "Longitude", "resale_price"]].dropna()
geo_s = geo.sample(min(18000, len(geo)), random_state=45)

plt.figure(figsize=(8, 8))
hb = plt.hexbin(
    geo_s["Longitude"],
    geo_s["Latitude"],
    C=geo_s["resale_price"],
    reduce_C_function=np.median,
    gridsize=55,
    cmap="magma",
    mincnt=3,
)
plt.colorbar(hb, label="Median resale_price in bin")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Median resale_price by location (hexbin, 18k sample)")
plt.tight_layout()
plt.show()



## 9. Takeaways (for modeling)

- **Target:** `resale_price` is right-skewed; consider `log1p(resale_price)` or metrics robust to skew; treat IQR outliers as sensitivity analysis rather than blind removal.
- **Time:** Use `Tranc_period` / year-month trends for drift and validation splits (e.g. time-based split).
- **Structure:** Strong role expected for `flat_type`, `floor_area_sqm`, `town` / `planning_area`, `hdb_age`, and storey proxies (`mid_storey`, `storey_range`).
- **Price normalisation:** `price_psm` helps compare across flat sizes; area interactions may matter.
- **Amenities:** Distance features (MRT, mall, hawker, schools) and within-radius flags may add signal; watch missingness on mall/hawker radius columns.
- **High cardinality:** `street_name`, `block`, school names — often better as clustering, target encoding after CV, or grouping.
- **Leakage caution:** Ensure test prediction setup matches features available at prediction time (same columns as `test.csv`).

Next steps: feature engineering notebook, baseline model (e.g. Gradient Boosting / LightGBM), and SHAP on key drivers.

